# Scenario Evaluation — Manual Reassignment Tool

Explore the impact of manual labor→driver reassignments on schedule KPIs.

**Section 1 — Scenario Editor**  
Load a base solution, inspect the assignment table, modify the `new_driver` column for
the labors you want to reassign, then save the result as a new scenario JSON file.

**Section 2 — Scenario Comparison**  
Load two or more saved scenario files and compare them side-by-side using standard KPI
tables, Gantt charts, distance figures, and route maps.

> **Sections are independent.** Run Section 2 without Section 1 by pointing
> `SCENARIO_PATHS` directly at pre-saved files.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path
from typing import List, Optional, Tuple

import pandas as pd
from IPython.display import display

# ── Project root on sys.path ────────────────────────────────────────────────
_PROJECT_ROOT = Path("../..").resolve()
if str(_PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT / "src"))

# ── Scenario editor helpers ─────────────────────────────────────────────────
from alfred.analysis.scenario_editor import (
    ScenarioSet,
    apply_reassignments,
    build_labor_editor_table,
    build_multi_scenario_overview_table,
    load_scenario_base,
    load_scenarios_for_comparison,
    reconstruct_scenario,
    save_scenario,
)

# ── Existing analysis utilities (reused as-is) ──────────────────────────────
from alfred.analysis.compare_solutions import (
    build_overview_table,
    load_and_prepare,
)
from alfred.analysis.solution_evaluation import (
    build_driver_distance_figure,
    build_gantt_figure,
    build_route_map,
    build_service_distance_figure,
    compute_payload_summary,
)
from alfred.optimization.settings.solver_settings import DEFAULT_DISTANCE_METHOD

print("Imports OK — project root:", _PROJECT_ROOT)

---
## Section 1 — Scenario Editor

Edit the configuration cell below, load the base solution, then modify `new_driver` in
the editor table before running the reconstruct and save cells.

In [ ]:
# ── Section 1 Configuration ─────────────────────────────────────────────────
# Required: path to the base output_payload.json you want to edit
BASE_PAYLOAD = _PROJECT_ROOT / "experiments" / "PLACEHOLDER" / "output" / "output_payload.json"

# Optional: driver directory JSON (enables home→first-service move distances)
DRIVER_DIRECTORY: Optional[Path] = None  # e.g. _PROJECT_ROOT / "data" / "driver_directory.json"

# Optional: input snapshot with full address data for recomputed labor distances
INPUT_FILE: Optional[Path] = None

# Filter to a single planning date (YYYY-MM-DD), or None to load all
PLANNING_DATE: Optional[str] = None  # e.g. "2026-05-05"

# Distance computation method (mirrors solver config)
DISTANCE_METHOD: str = DEFAULT_DISTANCE_METHOD

# Where to save the edited scenario
SCENARIO_OUTPUT_DIR: Path = _PROJECT_ROOT / "scenarios" / (PLANNING_DATE or "all_dates")
SCENARIO_LABEL: str = "scenario_v1"  # used as the filename and display label

print(f"BASE_PAYLOAD    : {BASE_PAYLOAD}")
print(f"SCENARIO_OUTPUT : {SCENARIO_OUTPUT_DIR / (SCENARIO_LABEL + '.json')}")
print(f"DISTANCE_METHOD : {DISTANCE_METHOD!r}")

In [ ]:
# ── Load base solution ───────────────────────────────────────────────────────
_base = load_scenario_base(
    payload_path=BASE_PAYLOAD,
    driver_directory_path=DRIVER_DIRECTORY,
    planning_date=PLANNING_DATE,
    distance_method=DISTANCE_METHOD,
    input_file=INPUT_FILE,
)

In [ ]:
# ── Base solution KPIs ───────────────────────────────────────────────────────
_base_summary = compute_payload_summary(
    _base.rows,
    tiempo_gracia_min=_base.params.tiempo_gracia_min,
    segments=_base.segments,
)
display(
    pd.DataFrame(
        [{"metric": k, "value": v} for k, v in _base_summary.items()]
    ).set_index("metric")
)

In [ ]:
# ── Build and display the editor table ──────────────────────────────────────
# Edit the `new_driver` column in the REASSIGNMENTS cell below.
editor_df = build_labor_editor_table(_base)
display(editor_df)

In [ ]:
# ── YOUR REASSIGNMENTS — edit this cell ─────────────────────────────────────
#
# Reassign a single labor:
#   editor_df.loc[editor_df["labor_id"] == 568175, "new_driver"] = "99"
#
# Reassign all labors of a service:
#   editor_df.loc[editor_df["service_id"] == 462197, "new_driver"] = "211"
#
# Reassign multiple labors at once:
#   editor_df.loc[editor_df["labor_id"].isin([568175, 568176]), "new_driver"] = "99"
#
# ─────────────────────────────────────────────────────────────────────────────

# (add your reassignment lines here)

# ─────────────────────────────────────────────────────────────────────────────
# Preview what will change:
_changed = editor_df[editor_df["new_driver"].astype(str) != editor_df["original_driver"].astype(str)]
print(f"{len(_changed)} labor(s) will be reassigned:")
display(_changed[["labor_id", "service_id", "labor_type", "original_driver", "new_driver"]])

In [ ]:
# ── Validate reassignments and reconstruct timeline ──────────────────────────
_orig_driver_map = dict(zip(editor_df["labor_id"], editor_df["original_driver"]))
_reassignment_dict = {
    row["labor_id"]: row["new_driver"]
    for _, row in editor_df.iterrows()
    if str(row["new_driver"]) != str(_orig_driver_map.get(row["labor_id"]))
}

print(f"Applying {len(_reassignment_dict)} reassignment(s)...")

_new_rows, _reassignment_warnings = apply_reassignments(_base.rows, _reassignment_dict)

if _reassignment_warnings:
    print(f"\n[apply_reassignments] {len(_reassignment_warnings)} warning(s):")
    for w in _reassignment_warnings:
        print(f"  ! {w}")

_new_rows, _new_segments = reconstruct_scenario(
    _new_rows,
    _base.points_lookup,
    _base.driver_home_lookup,
    _base.params,
    DISTANCE_METHOD,
)

_infeasible = [r for r in _new_rows if r.get("is_infeasible")]
if _infeasible:
    print(f"\n[infeasibility] {len(_infeasible)} labor(s) infeasible after reassignment:")
    _inf_df = pd.DataFrame(_infeasible)[
        ["labor_id", "service_id", "driver_id", "actual_start", "driver_move_distance_km"]
    ]
    display(_inf_df)
else:
    print("\n[infeasibility] No infeasible labors detected.")

In [ ]:
# ── Preview — Gantt chart of the reconstructed scenario ─────────────────────
_new_drivers = sorted({r["driver_id"] for r in _new_rows if r["driver_id"] is not None})
build_gantt_figure(_new_segments, _new_drivers, f"{SCENARIO_LABEL} (preview)").show()

In [ ]:
# ── Save scenario to JSON ────────────────────────────────────────────────────
_output_path = SCENARIO_OUTPUT_DIR / f"{SCENARIO_LABEL}.json"
_saved_path = save_scenario(
    original_services=_base.services,
    rows=_new_rows,
    output_path=_output_path,
    scenario_label=SCENARIO_LABEL,
)
print(f"Scenario saved: {_saved_path}")

---
## Section 2 — Scenario Comparison

Load two or more saved scenario JSON files and compare their KPIs.

- The **first entry** in `SCENARIO_PATHS` is used as the baseline for Δ columns.
- Section 2 is fully independent — run it without Section 1 by pointing
  `SCENARIO_PATHS` at any pre-saved files.
- Any saved scenario (from `save_scenario`) can also be used directly in
  `compare_solutions.ipynb` as `SOL_A_PAYLOAD` or `SOL_B_PAYLOAD`.

In [ ]:
# ── Section 2 Configuration ──────────────────────────────────────────────────
# List of (path, label) tuples. First entry = baseline for Δ columns.
SCENARIO_PATHS: List[Tuple[Path, str]] = [
    (BASE_PAYLOAD, "base"),
    # Add more scenarios:
    # (_PROJECT_ROOT / "scenarios" / "2026-05-05" / "scenario_v1.json", "scenario_v1"),
    # (_PROJECT_ROOT / "scenarios" / "2026-05-05" / "scenario_v2.json", "scenario_v2"),
]

# These can be set independently of Section 1
S2_DRIVER_DIRECTORY: Optional[Path] = DRIVER_DIRECTORY
S2_PLANNING_DATE: Optional[str] = PLANNING_DATE
S2_DISTANCE_METHOD: str = DISTANCE_METHOD
S2_INPUT_FILE: Optional[Path] = INPUT_FILE

print(f"Scenarios to compare: {[label for _, label in SCENARIO_PATHS]}")

In [ ]:
# ── Load all scenarios ───────────────────────────────────────────────────────
_scenario_set = load_scenarios_for_comparison(
    scenario_paths=SCENARIO_PATHS,
    driver_directory_path=S2_DRIVER_DIRECTORY,
    planning_date=S2_PLANNING_DATE,
    distance_method=S2_DISTANCE_METHOD,
    input_file=S2_INPUT_FILE,
)

In [ ]:
# ── KPI Overview Table (N scenarios) ────────────────────────────────────────
_overview = build_multi_scenario_overview_table(
    _scenario_set,
    baseline_label=SCENARIO_PATHS[0][1],
)
display(
    _overview.style
    .format(precision=2, na_rep="—")
    .set_caption(f"Scenario Comparison — Δ columns vs '{SCENARIO_PATHS[0][1]}'")
)

In [ ]:
# ── Pairwise comparison — runs only when exactly 2 scenarios are loaded ──────
# Uses the full compare_solutions pipeline for richer side-by-side figures.
if len(SCENARIO_PATHS) == 2:
    _la, _lb = SCENARIO_PATHS[0][1], SCENARIO_PATHS[1][1]
    _pair = load_and_prepare(
        sol_a=SCENARIO_PATHS[0][0],
        sol_b=SCENARIO_PATHS[1][0],
        driver_directory=S2_DRIVER_DIRECTORY,
        planning_date=S2_PLANNING_DATE,
        distance_method=S2_DISTANCE_METHOD,
        input_file=S2_INPUT_FILE,
    )
    print("\n── Pairwise overview (compare_solutions) ──")
    display(build_overview_table(_pair, _la, _lb))
else:
    print(f"Pairwise cell skipped ({len(SCENARIO_PATHS)} scenarios loaded). "
          "See multi-scenario charts below.")

In [ ]:
# ── Gantt Charts — one per scenario ─────────────────────────────────────────
for label, rows, segs in zip(
    _scenario_set.labels,
    _scenario_set.rows_list,
    _scenario_set.segments_list,
):
    _drivers = sorted({r["driver_id"] for r in rows if r["driver_id"] is not None})
    build_gantt_figure(segs, _drivers, label).show()

In [ ]:
# ── Distance per Service — one chart per scenario ────────────────────────────
for label, rows in zip(_scenario_set.labels, _scenario_set.rows_list):
    build_service_distance_figure(rows, _scenario_set.all_services, label).show()

In [ ]:
# ── Distance per Driver — one chart per scenario ─────────────────────────────
for label, rows in zip(_scenario_set.labels, _scenario_set.rows_list):
    _drivers = sorted({r["driver_id"] for r in rows if r["driver_id"] is not None})
    build_driver_distance_figure(rows, _drivers, label).show()

In [ ]:
# ── Route Maps — driver dropdown, one map per scenario ───────────────────────
import ipywidgets as widgets

_drv_dropdown = widgets.Dropdown(
    options=_scenario_set.all_drivers,
    description="Driver:",
    layout=widgets.Layout(width="300px"),
)
_map_output = widgets.Output()


def _render_maps(change):
    _map_output.clear_output(wait=True)
    selected_driver = change["new"]
    with _map_output:
        for label, rows in zip(_scenario_set.labels, _scenario_set.rows_list):
            if not any(r["driver_id"] == str(selected_driver) for r in rows):
                display(widgets.HTML(f"<p><b>{label}</b>: driver {selected_driver!r} has no labors.</p>"))
                continue
            home_wkt = (_scenario_set.driver_home_lookup or {}).get(str(selected_driver))
            display(widgets.HTML(f"<h4>{label}</h4>"))
            try:
                display(
                    build_route_map(
                        services=None,
                        rows=rows,
                        driver_id=selected_driver,
                        driver_home_wkt=home_wkt,
                        label=label,
                        points_lookup=_scenario_set.points_lookup,
                    )
                )
            except Exception as exc:
                display(widgets.HTML(f"<p style='color:red'>Map error: {exc}</p>"))


_drv_dropdown.observe(_render_maps, names="value")
_render_maps({"new": _drv_dropdown.value})
display(_drv_dropdown, _map_output)